# nb61 - Per-cell template-fit fractions (H23c)

**Error analysis.** nb59 confirmed the deblending mechanism: a clean-measured Grindhammer 2-blob fit recovers per-event pileup energy (partial corr +0.637 given sumE; signal-side 0.810). The champion model never sees this decomposition.

**Question.** Do the fit outputs - E_signal_fit, E_pileup_fit, chi2 improvement, N=2 flag - added as four global features improve the qd+EMA champion?

**Hypothesis.** H23b: the model uses the physics decomposition to sharpen contaminated events; win = >0.002 vs the nb58 anchor (singles 0.0418 +/- 0.0005) overall or in any E>17 bin. All template parameters from clean data; fits are deterministic per window; no new tunables.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from scipy.optimize import minimize as spmin
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution, PITCH, EPS
from picocal_data import build_grid, make_windows, splits_for, THRESH
from picocal_models import SubNetFQ, QUANTILES, width_binned_calibration, CFG
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB61_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB61_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
print(f'device {DEVICE} | mode {MODE} | build {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events
device cuda | mode full | build 136s


In [2]:
W = 4
def prof_fit():
    out = {}
    for reg in range(len(PITCH)):
        pts = []
        for ev in CE:
            if ev['reg'] != reg: continue
            m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
            if m.sum() < 4: continue
            e = ev['e'][m]; di = ev['di'][m].astype(float); dj = ev['dj'][m].astype(float)
            se = e.sum(); cx = (e * di).sum() / se; cy = (e * dj).sum() / se
            pts.append(np.stack([di - cx, dj - cy, e / se], 1))
        if len(pts) < 50: continue
        P = np.concatenate(pts)
        def p2d(dx, dy, R):
            r2 = dx ** 2 + dy ** 2
            return R ** 2 / (np.pi * (r2 + R ** 2) ** 2)
        def loss(th):
            return float(((p2d(P[:, 0], P[:, 1], np.exp(th[0])) - P[:, 2]) ** 2).sum())
        res = spmin(loss, [np.log(0.6)], method='Nelder-Mead', options=dict(maxiter=500))
        out[reg] = float(np.exp(res.x[0]))
    fb = np.mean(list(out.values())) if out else 0.6
    for reg in range(len(PITCH)):
        if reg not in out: out[reg] = fb
    return out
RPROF = prof_fit()
print('profile R per region:', {int(PITCH[k]): round(v, 3) for k, v in RPROF.items()})
def fit_features(rows, regs):
    NB = len(rows); L = (2*W+1)**2
    EG = np.zeros((NB, L), np.float32); MASK = np.zeros((NB, L), np.float32)
    DI = np.zeros((NB, L), np.float32); DJ = np.zeros((NB, L), np.float32)
    SX = np.zeros(NB, np.float32); SY = np.zeros(NB, np.float32); RR = np.zeros(NB, np.float32)
    for k in range(NB):
        tok = rows[k][0]; n = tok.shape[0]
        e = np.expm1(tok[:, 0])
        EG[k, :n] = e; MASK[k, :n] = 1.0
        DI[k, :n] = tok[:, 3]; DJ[k, :n] = tok[:, 4]
        s = int(np.argmax(e)); SX[k] = tok[s, 3]; SY[k] = tok[s, 4]
        RR[k] = RPROF[regs[k]]
    tE = torch.tensor(EG, device=DEVICE); tM = torch.tensor(MASK, device=DEVICE)
    tDI = torch.tensor(DI, device=DEVICE); tDJ = torch.tensor(DJ, device=DEVICE)
    tR = torch.tensor(RR, device=DEVICE)[:, None]
    sx = torch.tensor(SX, device=DEVICE)[:, None]; sy = torch.tensor(SY, device=DEVICE)[:, None]
    sE = tE.sum(1, keepdim=True)
    def blob(cx, cy, lE):
        r2 = (tDI - cx) ** 2 + (tDJ - cy) ** 2
        return torch.exp(lE) * tR ** 2 / (np.pi * (r2 + tR ** 2) ** 2)
    def run_fit(nb_, steps=600):
        torch.manual_seed(0)
        d1 = torch.zeros(NB, 2, device=DEVICE, requires_grad=True)
        lE1 = torch.log(sE.squeeze(1) * 0.8 + 1.0).clone().requires_grad_(True)
        params = [d1, lE1]
        if nb_ == 2:
            resid0 = tE - blob(sx, sy, torch.log(sE * 0.8 + 1.0).squeeze(1)[:, None] * torch.ones_like(sx))
            far = torch.argmax((resid0 * tM) * (((tDI - sx) ** 2 + (tDJ - sy) ** 2) > 2).float(), 1)
            c2 = torch.stack([tDI.gather(1, far[:, None]).squeeze(1),
                              tDJ.gather(1, far[:, None]).squeeze(1)], 1).clone().requires_grad_(True)
            lE2 = torch.log(sE.squeeze(1) * 0.2 + 1.0).clone().requires_grad_(True)
            params += [c2, lE2]
        opt = torch.optim.Adam(params, lr=0.05)
        for _ in range(steps):
            opt.zero_grad()
            pred = blob(sx + torch.tanh(d1[:, 0:1]), sy + torch.tanh(d1[:, 1:2]), lE1[:, None])
            if nb_ == 2:
                pred = pred + blob(c2[:, 0:1], c2[:, 1:2], lE2[:, None])
            chi = (((pred - tE) ** 2) * tM / (tE.abs() + 10.0)).sum(1)
            chi.mean().backward(); opt.step()
        with torch.no_grad():
            pred1 = blob(sx + torch.tanh(d1[:, 0:1]), sy + torch.tanh(d1[:, 1:2]), lE1[:, None])
            pred = pred1
            frac = torch.zeros_like(pred1)
            if nb_ == 2:
                pred2 = blob(c2[:, 0:1], c2[:, 1:2], lE2[:, None])
                pred = pred1 + pred2
                frac = pred2 / (pred1 + pred2 + 1e-9)
            chi = (((pred - tE) ** 2) * tM / (tE.abs() + 10.0)).sum(1)
            return chi.cpu().numpy(), torch.exp(lE1).cpu().numpy(), frac.cpu().numpy()
    chi1, E1a, fr1 = run_fit(1)
    chi2_, E1b, fr2 = run_fit(2)
    use2 = chi2_ < 0.95 * chi1
    return np.where(use2[:, None], fr2, 0.0).astype(np.float32)
t1 = time.time()
rows_m, keep_m = make_windows(W, ME)
ktr, kva, kte = splits_for(keep_m, len(ME))
regs_m = [r[4] for r in rows_m]
F_m = fit_features(rows_m, regs_m)
rows_c, _ = make_windows(W, CE)
F_c = fit_features(rows_c, [r[4] for r in rows_c])
FITF = np.concatenate([F_m, F_c]).astype(np.float32)
print(f'template fits for {len(FITF)} windows in {time.time()-t1:.0f}s')

profile R per region: {15: 0.613, 30: 0.604, 40: 0.618, 60: 0.597, 120: 0.578}


template fits for 102857 windows in 20s


In [3]:
NG = 6
rows = rows_m + rows_c
n_mb = len(rows_m); ctr = np.arange(n_mb, len(rows))
N = len(rows); L = (2*W+1)**2; IN_DIM = rows[0][0].shape[1]
y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
Et = np.array([r[3] for r in rows], np.float32)
sumE = np.array([r[1] for r in rows], np.float32)
X = np.zeros((N, L, IN_DIM + 1), np.float32); M = np.zeros((N, L), np.bool_)
G = np.zeros((N, NG), np.float32); Eraw = np.zeros((N, L), np.float32)
for i, (tok, se, sde, et, rg, etv) in enumerate(rows):
    n = tok.shape[0]; X[i, :n, :IN_DIM] = tok; X[i, :n, IN_DIM] = FITF[i, :n]; M[i, :n] = True
    e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
    lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
    fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
    G[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat, float(i >= n_mb)]
la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
G[:, :5] = (G[:, :5] - G[ktr, :5].mean(0)) / (G[ktr, :5].std(0) + EPS)
NC = 9
cont2 = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
mean = cont2.mean(0); std = cont2.std(0) + EPS
X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
T = dict(X=torch.from_numpy(X).to(DEVICE), M=torch.from_numpy(M).to(DEVICE),
         G=torch.from_numpy(G).to(DEVICE), Y=torch.from_numpy(y).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(Eraw).to(DEVICE))
QS = torch.tensor(QUANTILES, device=DEVICE)
print(f'N {N}, NG {NG}, tr/va/te {len(ktr)}/{len(kva)}/{len(kte)}')

N 102857, NG 6, tr/va/te 50787/10883/10884


In [4]:
LAM_QD = 0.5
TAU = 0.02
def qd_loss(q, yb):
    d = yb - q
    pin = torch.maximum(QS * d, (QS - 1) * d).mean()
    width = (q[:, 2] - q[:, 1]).abs() + (q[:, 1] - q[:, 0]).abs()
    inside = torch.sigmoid((yb.squeeze(1) - q[:, 0]) / TAU) * torch.sigmoid((q[:, 2] - yb.squeeze(1)) / TAU)
    return pin + LAM_QD * (width.mean() + 10.0 * torch.relu(0.5 - inside.mean()) ** 2)
def train_eval(seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetFQ(IN_DIM + 1, la0, lb0, ng=NG).to(DEVICE)
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tr_idx = np.concatenate([np.asarray(ktr), ctr])
    ck = CKPT / f'nb61_cellfrac_s{seed}.pt'
    def batches(idx, bs, sh=None):
        idx = np.asarray(idx)
        if sh is not None: idx = sh.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(m_, b): return m_(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def vloss(m_):
        m_.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256):
                d = T['Y'][b] - fwd(m_, b)
                s += torch.maximum(QS * d, (QS - 1) * d).mean().item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); ema.load_state_dict(st['ema'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume s{seed} from ep {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(tr_idx, CFG['batch'], rng):
            opt.zero_grad()
            qd_loss(fwd(model, b), T['Y'][b]).backward()
            opt.step()
            ema.update_parameters(model)
        sched.step()
        vv = vloss(ema.module)
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(ema.module.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opt=opt.state_dict(),
                        sched=sched.state_dict(), best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    final = SubNetFQ(IN_DIM + 1, la0, lb0, ng=NG).to(DEVICE)
    final.load_state_dict(bstate); final.eval()
    def run(idx):
        out = []
        with torch.no_grad():
            for b in batches(idx, 256): out.append(fwd(final, b).cpu().numpy())
        return np.concatenate(out)
    pe = width_binned_calibration(run(kva), run(kte), y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb61_cellfrac{TAG}.csv'
done = set()
if CSVP.exists():
    done = set(pd.read_csv(CSVP)['seed'])
    print('resume, done:', sorted(done))
for seed in SEEDS:
    if seed in done: print('skip', seed); continue
    t2 = time.time()
    sig, pe = train_eval(seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb61_pred{TAG}_cellfrac_s{seed}.npy', pe)
    row = dict(seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t2))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'cellfrac seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

cellfrac seed 0: sigma_eff 0.0415 (2764s)


cellfrac seed 1: sigma_eff 0.0423 (3696s)


 seed  sigma_eff  elapsed
    0     0.0415     2764
    1     0.0423     3696


## Verdict

Anchor: nb58 qd+EMA singles 0.0418 +/- 0.0005 (identical recipe minus the 4 fit features). Win = >0.002 overall or any E>17 bin; diagnostic = contamination tertiles (the features must help hiC).

In [5]:
te_e = Et[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
preds = [np.load(OUT / f'nb61_pred{TAG}_cellfrac_s{s}.npy') for s in SEEDS
         if (OUT / f'nb61_pred{TAG}_cellfrac_s{s}.npy').exists()]
if preds:
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    bins = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        bins.append(f'{resolution(ens[mm], te_e[mm])["sigma_eff"]:.4f}')
    cont = sumE[np.asarray(kte)] / np.maximum(1000.0 * te_e, EPS)
    qs2 = np.quantile(cont, [1/3, 2/3])
    terts = []
    for gsel, lab in [(cont < qs2[0], 'loC'), ((cont >= qs2[0]) & (cont < qs2[1]), 'miC'), (cont >= qs2[1], 'hiC')]:
        terts.append(f'{lab}:{resolution(ens[gsel], te_e[gsel])["sigma_eff"]:.4f}')
    print(f'cellfrac mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}  [anchor 0.0418 +/- 0.0005; nb58 hiC ref from stack ~0.073]')
    print('per-bin ' + ' / '.join(bins) + ' | ' + ' '.join(terts))

cellfrac mean 0.0419 +/- 0.0004 | ens 0.0413  [anchor 0.0418 +/- 0.0005; nb58 hiC ref from stack ~0.073]
per-bin 0.0651 / 0.0455 / 0.0340 / 0.0354 / 0.0343 / 0.0349 | loC:0.0274 miC:0.0367 hiC:0.0736
